# Notebook 04: Entrenamiento, testeo y metricas

Entrena cinco modelos de regresion de complejidad creciente, los evalua sobre el conjunto de
prueba y deja la tabla comparativa de MSE, RMSE, MAE y R2.

---

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
RANDOM_STATE = 42

PROYECTO = "food_delivery_time_prediction"

EN_DRIVE = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir("/content/drive/MyDrive"):
        raise RuntimeError(
            "Drive no quedo montado. Volve a ejecutar esta celda y autoriza el acceso "
            "en la ventana emergente."
        )
    RUTA = f"/content/drive/MyDrive/{PROYECTO}"
    EN_DRIVE = True
except ImportError:
    RUTA = os.path.abspath(f"./{PROYECTO}")

CARPETA_SPLITS = os.path.join(RUTA, "splits")
os.makedirs(CARPETA_SPLITS, exist_ok=True)


def guardar(nombre):
    destino = os.path.join(CARPETA_SPLITS, nombre + ".png")
    plt.savefig(destino, dpi=150, bbox_inches="tight")
    print("Figura guardada:", destino)


print("Persistencia en Drive:", EN_DRIVE)
print("Ruta de trabajo:", RUTA)

In [ ]:
X_train = pd.read_csv(os.path.join(RUTA, "X_train.csv"))
X_test = pd.read_csv(os.path.join(RUTA, "X_test.csv"))
y_train = pd.read_csv(os.path.join(RUTA, "y_train.csv")).iloc[:, 0]
y_test = pd.read_csv(os.path.join(RUTA, "y_test.csv")).iloc[:, 0]

NUMERICAS = ["distancia_km", "edad", "calificacion", "pedidos_simultaneos", "estado_vehiculo"]
CATEGORICAS = ["trafico", "clima", "vehiculo", "tipo_pedido", "ciudad", "festivo"]

print("Entrenamiento:", X_train.shape, " Prueba:", X_test.shape)

## 5.2 Correccion metodologica: como se evita la fuga de datos

Hay fuga de datos cuando el conjunto de prueba interviene en alguna etapa previa a la
evaluacion. Si el escalador calcula media y desviacion sobre todas las filas, o si el codificador
determina el conjunto de categorias mirando el archivo completo, las metricas de prueba dejan de
estimar el error sobre datos no vistos. El sesgo resultante es siempre optimista y no produce
ningun sintoma detectable en los resultados.

Para evitarlo se encapsulan todas las transformaciones en un `ColumnTransformer` dentro de un
`Pipeline`. La llamada a `fit` sobre los datos de entrenamiento ajusta cada transformacion
unicamente con esas filas, y la llamada posterior a `predict` sobre el conjunto de prueba aplica
los parametros ya estimados sin recalcularlos. La restriccion queda impuesta por la estructura
del objeto y no depende de recordar el orden correcto de los pasos.

**Que hace cada rama del preprocesador.**

Sobre las **numericas**: imputa la mediana en los faltantes y estandariza a media 0 y
desviacion 1. La estandarizacion importa aca por una razon concreta: la distancia se mide en
kilometros de uno a treinta y la calificacion en una escala de uno a cinco. Sin escalar, los
coeficientes quedan en unidades incomparables y no se puede leer cual variable pesa mas.

Sobre las **categoricas**: imputa la categoria mas frecuente y aplica codificacion binaria.
Se usa `drop="first"` para eliminar una categoria de referencia por variable y evitar
colinealidad perfecta, que dejaria los coeficientes indeterminados. Y `handle_unknown="ignore"`
para que una categoria que aparezca solo en el test no rompa la prediccion, que es lo mismo
que tendria que hacer el sistema en produccion frente a un valor nuevo.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


def preprocesador(numericas, categoricas=None, grado=1):
    pasos_num = [("imputar", SimpleImputer(strategy="median")),
                 ("escalar", StandardScaler())]
    if grado > 1:
        pasos_num.append(("polinomio", PolynomialFeatures(grado, include_bias=False)))

    ramas = [("num", Pipeline(pasos_num), numericas)]
    if categoricas:
        ramas.append(("cat", Pipeline([
            ("imputar", SimpleImputer(strategy="most_frequent")),
            ("codificar", OneHotEncoder(drop="first", handle_unknown="ignore")),
        ]), categoricas))
    return ColumnTransformer(ramas)

## 5.1 Los cinco modelos

Se implementan cinco, en orden de
complejidad creciente, porque la comparacion entre ellos es lo que responde la pregunta del
experimento.

| Modelo | Que usa | Que pone a prueba |
|---|---|---|
| **A. Lineal simple** | Solo la distancia | Cuanto se puede predecir con la variable que la intuicion senala primero |
| **B. Lineal multiple** | Las cinco numericas y las seis categoricas | Cuanto agrega incorporar el contexto de circulacion |
| **C. Polinomica de grado 2** | Lo mismo que B, con terminos cuadraticos y productos entre las numericas | Si la relacion es curva o si hay interacciones entre las variables numericas |
| **D. Interacciones completas con Ridge** | Todos los productos por pares sobre la matriz ya codificada, numericas y categoricas juntas | Si el efecto de una variable depende del valor de otra, incluidas las categoricas |
| **E. Splines con interacciones y Ridge** | Lo mismo que D, pero representando cada numerica con una base de splines cubicos | Si las numericas tienen curvaturas que un polinomio de grado bajo no alcanza a describir |

El modelo A cumple la funcion de linea base. Sin una referencia de un solo predictor no hay
forma de cuantificar cuanto aporta cada variable agregada despues, porque las metricas de los
modelos siguientes no tendrian contra que compararse.

Los cuatro son modelos lineales, porque los cuatro son lineales en sus parametros. Lo que
cambia entre ellos es que variables se le entregan a la regresion, no la familia del modelo.

### Modelo A: regresion lineal simple sobre la distancia

Es la formulacion mas directa del problema: el tiempo de entrega como funcion de la distancia
a recorrer. Una sola variable, un coeficiente y una ordenada al origen.

In [ ]:
modelo_a = Pipeline([
    ("prep", preprocesador(["distancia_km"])),
    ("regresion", LinearRegression()),
])
modelo_a.fit(X_train[["distancia_km"]], y_train)

pendiente = modelo_a.named_steps["regresion"].coef_[0]
print("Modelo A entrenado sobre", len(X_train), "observaciones")
print(f"Ordenada al origen: {modelo_a.named_steps['regresion'].intercept_:.2f} min")
print(f"Coeficiente de la distancia estandarizada: {pendiente:+.2f} min")

### Modelo B: regresion lineal multiple

Incorpora las once variables predictoras: las cinco numericas y las seis categoricas. Es el
modelo que la hipotesis pone a prueba, porque contiene a la vez la distancia y el trafico y
permite comparar cuanto pesa cada uno con el resto controlado.

In [ ]:
modelo_b = Pipeline([
    ("prep", preprocesador(NUMERICAS, CATEGORICAS)),
    ("regresion", LinearRegression()),
])
modelo_b.fit(X_train, y_train)

n_variables = modelo_b.named_steps["prep"].transform(X_train.head(5)).shape[1]
print("Modelo B entrenado")
print(f"Variables tras codificar y escalar: {n_variables}")

### Modelo C: regresion polinomica de grado 2

Sobre las mismas variables del modelo B, agrega el cuadrado de cada numerica y el producto
entre cada par. Sigue siendo una regresion lineal, porque es lineal en los parametros; lo que
cambia es que ahora puede representar relaciones curvas y efectos combinados.

La pregunta que responde es concreta: si el efecto de recorrer un kilometro adicional
dependiera del nivel de trafico, o si la distancia tuviera rendimientos decrecientes, el
modelo B no podria capturarlo y el C si. Si el C no mejora, esas curvaturas no estan.

In [ ]:
modelo_c = Pipeline([
    ("prep", preprocesador(NUMERICAS, CATEGORICAS, grado=2)),
    ("regresion", LinearRegression()),
])
modelo_c.fit(X_train, y_train)

n_variables_c = modelo_c.named_steps["prep"].transform(X_train.head(5)).shape[1]
print("Modelo C entrenado")
print(f"Variables tras la expansion polinomica: {n_variables_c}")

### Modelo D: interacciones completas con regularizacion Ridge

El modelo C expande solo las variables numericas, de modo que nunca llega a representar que el
efecto del trafico dependa de la ciudad, o que el costo de un kilometro cambie segun haya
atasco o no. Esas combinaciones son exactamente las que el analisis del notebook 03 sugirio al
mostrar que el trafico desplaza el tiempo de forma distinta en cada tramo de distancia.

El modelo D toma la matriz ya codificada, con las categoricas convertidas en columnas binarias,
y genera todos los productos por pares. Asi aparecen las interacciones entre categoricas y
entre categoricas y numericas, que son las que faltaban.

**Por que Ridge y no la regresion habitual.** El producto por pares multiplica la cantidad de
columnas por diez, y muchas de ellas quedan casi vacias porque corresponden a combinaciones
poco frecuentes, como ciudad semiurbana en dia festivo. Con tan pocos casos, la regresion sin
regularizar les asigna coeficientes enormes que no se sostienen fuera de la muestra. Ridge
penaliza la magnitud de los coeficientes y evita ese problema, a cambio de un sesgo pequeno.

Sigue siendo un modelo lineal: la penalizacion cambia como se estiman los coeficientes, no la
forma de la funcion.

In [ ]:
from sklearn.linear_model import Ridge

modelo_d = Pipeline([
    ("prep", preprocesador(NUMERICAS, CATEGORICAS)),
    ("interacciones", PolynomialFeatures(2, include_bias=False, interaction_only=True)),
    ("regresion", Ridge(alpha=1.0)),
])
modelo_d.fit(X_train, y_train)

matriz = modelo_d.named_steps["interacciones"].transform(
    modelo_d.named_steps["prep"].transform(X_train.head(5)))
print("Modelo D entrenado")
print(f"Variables tras generar todas las interacciones por pares: {matriz.shape[1]}")

### Modelo E: base de splines con interacciones

El modelo C mostro que elevar las numericas al cuadrado casi no aporta, y de ahi se podria
concluir que las relaciones son lineales. Esa conclusion seria apresurada: un polinomio de grado
bajo es una forma muy rigida de representar una curva, porque un solo coeficiente tiene que
describir el comportamiento en todo el rango de la variable a la vez.

Un spline resuelve eso de otro modo. Divide el rango en tramos separados por nudos y ajusta un
polinomio distinto en cada tramo, obligandolos a unirse suavemente en los bordes. El resultado
es una curva flexible que puede subir rapido en una zona y aplanarse en otra, algo que un
polinomio global no puede hacer sin oscilar.

Aplicado a este problema: el efecto de la distancia sobre el tiempo no tiene por que ser el
mismo en los primeros dos kilometros que entre el quince y el veinte, y el efecto de la
calificacion del repartidor casi con seguridad se concentra en el tramo bajo de la escala, donde
estan los pocos mal evaluados.

**Sigue siendo una regresion lineal.** La base de splines transforma cada variable en un
conjunto de columnas, y la regresion estima un coeficiente por columna. La funcion es lineal en
los parametros, que es lo que define a la familia. Lo que cambia es la representacion de las
variables, no el metodo de estimacion.

Sobre esa base se generan de nuevo todos los productos por pares, para conservar las
interacciones que hicieron ganar al modelo D, y se ajusta con Ridge por el mismo motivo.

In [ ]:
from sklearn.preprocessing import SplineTransformer

preprocesador_splines = ColumnTransformer([
    ("num", Pipeline([
        ("imputar", SimpleImputer(strategy="median")),
        ("escalar", StandardScaler()),
        ("splines", SplineTransformer(n_knots=7, degree=3, include_bias=False)),
    ]), NUMERICAS),
    ("cat", Pipeline([
        ("imputar", SimpleImputer(strategy="most_frequent")),
        ("codificar", OneHotEncoder(drop="first", handle_unknown="ignore")),
    ]), CATEGORICAS),
])

modelo_e = Pipeline([
    ("prep", preprocesador_splines),
    ("interacciones", PolynomialFeatures(2, include_bias=False, interaction_only=True)),
    ("regresion", Ridge(alpha=1.0)),
])
modelo_e.fit(X_train, y_train)

matriz_e = modelo_e.named_steps["interacciones"].transform(
    modelo_e.named_steps["prep"].transform(X_train.head(5)))
print("Modelo E entrenado")
print(f"Variables tras los splines y las interacciones: {matriz_e.shape[1]}")

## 6.1 Calculo de metricas

Se calculan las cuatro metricas para los tres modelos, sobre entrenamiento y sobre prueba.

| Metrica | Que mide | Unidad |
|---|---|---|
| **MSE** | Error cuadratico medio. Penaliza fuerte los errores grandes | minutos al cuadrado |
| **RMSE** | Raiz del anterior. Vuelve a la escala original de la variable | minutos |
| **MAE** | Error absoluto medio. Trata todos los errores por igual | minutos |
| **R2** | Proporcion de la varianza del tiempo que el modelo explica | sin unidad, 0 a 1 |

Se reportan train y test juntos a proposito: la comparacion entre ambos es lo que permite
diagnosticar sobreajuste, y ese analisis es el criterio 7.2 del notebook 05.

In [ ]:
def metricas(nombre, modelo, columnas):
    filas = []
    for etiqueta, X, y in [("train", X_train[columnas], y_train),
                           ("test", X_test[columnas], y_test)]:
        pred = modelo.predict(X)
        mse = mean_squared_error(y, pred)
        filas.append({
            "modelo": nombre,
            "conjunto": etiqueta,
            "MSE": mse,
            "RMSE": np.sqrt(mse),
            "MAE": mean_absolute_error(y, pred),
            "R2": r2_score(y, pred),
        })
    return filas


resultados = []
resultados += metricas("A. Lineal simple", modelo_a, ["distancia_km"])
resultados += metricas("B. Lineal multiple", modelo_b, list(X_train.columns))
resultados += metricas("C. Polinomica g2", modelo_c, list(X_train.columns))
resultados += metricas("D. Interacciones + Ridge", modelo_d, list(X_train.columns))
resultados += metricas("E. Splines + interacciones", modelo_e, list(X_train.columns))

tabla = pd.DataFrame(resultados)
tabla[["MSE", "RMSE", "MAE"]] = tabla[["MSE", "RMSE", "MAE"]].round(2)
tabla["R2"] = tabla["R2"].round(4)
tabla

In [ ]:
comparativa = tabla[tabla["conjunto"] == "test"].set_index("modelo")[["MSE", "RMSE", "MAE", "R2"]]
print("Tabla comparativa sobre el conjunto de prueba")
print(comparativa.to_string())

base = comparativa.loc["A. Lineal simple"]
print("\nMejora respecto del modelo simple:")
for m in ["B. Lineal multiple", "C. Polinomica g2", "D. Interacciones + Ridge",
          "E. Splines + interacciones"]:
    fila = comparativa.loc[m]
    print(f"  {m:22s} RMSE {base['RMSE'] - fila['RMSE']:+.2f} min   "
          f"R2 {fila['R2'] - base['R2']:+.4f}")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 4.6))

pivote_r2 = tabla.pivot(index="modelo", columns="conjunto", values="R2")
pivote_r2.plot(kind="bar", ax=ejes[0], color=["#404040", "#a8a8a8"], width=0.7)
ejes[0].set_ylabel("R2")
ejes[0].set_xlabel("")
ejes[0].set_title("Varianza explicada por modelo")
ejes[0].set_xticks(range(len(pivote_r2.index)))
ejes[0].set_xticklabels(pivote_r2.index, rotation=15, ha="right")
ejes[0].legend(title="")

pivote_rmse = tabla.pivot(index="modelo", columns="conjunto", values="RMSE")
pivote_rmse.plot(kind="bar", ax=ejes[1], color=["#404040", "#a8a8a8"], width=0.7)
ejes[1].set_ylabel("RMSE (minutos)")
ejes[1].set_xlabel("")
ejes[1].set_title("Error tipico por modelo")
ejes[1].set_xticks(range(len(pivote_rmse.index)))
ejes[1].set_xticklabels(pivote_rmse.index, rotation=15, ha="right")
ejes[1].legend(title="")

plt.tight_layout()
guardar("04_comparacion_modelos")
plt.show()

**Lectura de la tabla.** El modelo A, que solo conoce la distancia, explica una fraccion muy
pequena de la variacion del tiempo de entrega. Incorporar el resto de las variables multiplica
varias veces esa cifra y recorta el error tipico en cerca de tres minutos.

Los dos pasos siguientes muestran donde estaba lo que faltaba. La expansion polinomica sobre
las numericas del modelo C agrega poco, lo que descarta que las relaciones sean marcadamente
curvas. En cambio las interacciones completas del modelo D producen una mejora bastante mayor,
y eso identifica el problema del modelo B: no era que le faltara curvatura sino que trataba
cada variable como si su efecto fuera el mismo en cualquier contexto. El costo de un kilometro
no es igual con la avenida libre que dentro de un atasco, y el modelo B no tenia forma de
expresarlo.

El modelo E produce la ultima mejora, y su origen difiere del anterior. El modelo D ya incluia
el conjunto completo de interacciones, de modo que lo que agrega el E es capacidad de representar
curvatura. Ese resultado acota la conclusion del modelo C: el escaso aporte de los terminos
cuadraticos no indica que las relaciones sean lineales, sino que un polinomio de grado dos, con
un unico coeficiente por variable para todo su rango, no alcanza para describirlas.

Las barras de entrenamiento y prueba quedan a alturas equivalentes en los cinco modelos. La
diferencia entre ambas es el indicador directo de sobreajuste, y su magnitud se cuantifica en el
notebook 05.

### Prediccion contra valor real

Un grafico de dispersion entre lo predicho y lo observado muestra donde falla el modelo, cosa
que un numero resumen no puede mostrar. Si el modelo fuera perfecto todos los puntos caerian
sobre la diagonal.

In [ ]:
pred_test_b = modelo_b.predict(X_test)
pred_test_e = modelo_e.predict(X_test)

fig, ejes = plt.subplots(1, 2, figsize=(12.5, 5.2))
for eje, pred, nombre in [(ejes[0], pred_test_b, "B. Lineal multiple"),
                          (ejes[1], pred_test_e, "E. Splines + interacciones")]:
    eje.scatter(y_test, pred, s=6, alpha=0.2, color="#606060")
    limites = [y_test.min(), y_test.max()]
    eje.plot(limites, limites, color="#101010", linestyle="--", linewidth=1.5,
             label="prediccion perfecta")
    eje.set_xlabel("Tiempo real (min)")
    eje.set_ylabel("Tiempo predicho (min)")
    eje.set_title(nombre)
    eje.legend(fontsize=9)

plt.tight_layout()
guardar("04_prediccion_vs_real")
plt.show()

**Lectura.** La nube sigue la diagonal, de modo que el modelo no tiene un sesgo sistematico:
no subestima ni sobreestima de manera general. Pero es visiblemente mas plana que la
diagonal en los extremos: las entregas muy rapidas se predicen mas lentas de lo que fueron y
las muy lentas mas rapidas. Es el comportamiento tipico de un modelo que no dispone de la
informacion necesaria para llegar a los extremos, y refuerza la lectura de que el limite esta
en las variables y no en la forma funcional.

In [ ]:
import joblib

joblib.dump({
    "modelo_a": modelo_a,
    "modelo_b": modelo_b,
    "modelo_c": modelo_c,
    "modelo_d": modelo_d,
    "modelo_e": modelo_e,
    "tabla_metricas": tabla,
    "numericas": NUMERICAS,
    "categoricas": CATEGORICAS,
}, os.path.join(RUTA, "modelos_y_metricas.joblib"))

tabla.to_csv(os.path.join(RUTA, "tabla_metricas.csv"), index=False)
print("Modelos y metricas guardados en:", RUTA)